# IDX Exchange DS Internship - Week Nine -  Simple Prediction App (Streamlit)

* Build a Streamlit app: user inputs LivingArea, Beds, Baths, LotSize → output predicted price.
* Load trained model with joblib/pickle.

---------

This week we got to construct a simple interactive app that would allow users to input certain property metrics and receive a predicted close price. In order to use the app, a model ("model.joblib") must be trained and then loaded onto the streamlit script. In this notebook, a model is trained on LivingArea, LotSizeSquareFeet, BathroomsTotalInteger, BedroomsTotal, and City to predict ClosePrice. Becuase target encoding is performed on City, the encoder ("city_encoder.joblib") must also be saved and uploaded to the streamlit script. Once the model is trained, it is also evaluation to see how this simplified version compares to the finalized model.

In [26]:
import joblib
import pandas as pd
import xgboost as xgb
from xgboost import XGBRegressor
import numpy as np
from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.metrics import mean_absolute_error

from sklearn.preprocessing import TargetEncoder

import warnings
warnings.filterwarnings("ignore")

## Import Data

In [15]:
# Create functions for quickly evaluating the models

def median_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.median(np.abs((y_true - y_pred) / y_true))

def metrics(y_true, y_pred, logged=True):
    # Note: you have to undo the log transform on ClosePrice by using np.exp()
    if logged == True:
        y_true = np.exp(y_true)
        y_pred = np.exp(y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    mdape = median_absolute_percentage_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    return r2, mape * 100, mdape * 100, mae

def metric_df(r2_train, mape_train, mdape_train, mae_train,
              r2_val, mape_val, mdape_val, mae_val,
              r2_test, mape_test, mdape_test, mae_test):
    data = {"r2": [r2_train, r2_val, r2_test],
            "MAPE": [mape_train, mape_val, mape_test],
            "MdAPE": [mdape_train, mdape_val, mdape_test],
            "MAE": [f"${round(mae_train, 2)}", f"${round(mae_val, 2)}", f"${round(mae_test, 2)}"]}
    df = pd.DataFrame(data)
    df.index = ["Train Set", "Validation Set", "Test Set"]
    return df

In [19]:
# Read in the training and test sets created in week six

train_and_val = pd.read_csv("feat_eng_train.csv")
testing = pd.read_csv("feat_eng_test.csv")

## Target Encoding

In [20]:
# Target encode City so that users can input city into the app
# Saving the encoder ensures that the streamlit model has access to the correct encodings

# Define columns to encode. Just City in this case
to_encode_train = train_and_val[["City"]]
to_encode_test = testing[["City"]]

encoder = TargetEncoder()
x_train_encoded = encoder.fit_transform(to_encode_train, train_and_val["ClosePrice"])
x_test_encoded = encoder.transform(to_encode_test)

x_train_encoded_df = pd.DataFrame(x_train_encoded, columns=to_encode_train.columns)
x_test_encoded_df = pd.DataFrame(x_test_encoded, columns=to_encode_train.columns)

# train_and_val and testing already have a "City_enc" column from previous weeks. Drop it for this notebook
train_and_val = train_and_val.drop(columns=["City_enc"])
testing = testing.drop(columns=["City_enc"])

train_and_val = train_and_val.join(x_train_encoded_df, rsuffix='_enc')
testing = testing.join(x_test_encoded_df, rsuffix='_enc')

## Train Model

In [21]:
# Carve out training and validation sets
training = train_and_val[~(train_and_val["PropertyMonth"] == "2026-03-01")]
validation = train_and_val[train_and_val["PropertyMonth"] == "2026-03-01"]

print(train_and_val.shape, training.shape, validation.shape)

(265296, 153) (255004, 153) (10292, 153)


In [22]:
common_cols = ["LivingArea", "LotSizeSquareFeet",
               "BathroomsTotalInteger", "BedroomsTotal",
               "City_enc"]

x_train = training[common_cols]
y_train = training["ClosePrice"]

x_val = validation[common_cols]  
y_val = validation["ClosePrice"]

x_test = testing[common_cols]
y_test = testing["ClosePrice"]

In [27]:
# Train and evaluate model

xgboo = xgb.XGBRegressor(n_estimators=900, learning_rate=0.05, max_depth=10, random_state=42)
xgboo.fit(x_train, y_train)

pred_train_xgb = xgboo.predict(x_train)
pred_val_xgb = xgboo.predict(x_val)
pred_test_xgb = xgboo.predict(x_test)

r2_train, mape_train, mdape_train, mae_train = metrics(y_train, pred_train_xgb, False)
r2_val, mape_val, mdape_val, mae_val = metrics(y_val, pred_val_xgb, False)
r2_test, mape_test, mdape_test, mae_test = metrics(y_test, pred_test_xgb, False)

xgboost_model = metric_df(r2_train, mape_train, mdape_train, mae_train, 
                        r2_val, mape_val, mdape_val, mae_val,
                        r2_test, mape_test, mdape_test, mae_test)
xgboost_model

,r2,MAPE,MdAPE,MAE
Train Set,0.905422,11.278113,7.987053,$119365.38
Validation Set,0.816141,14.586557,10.071690,$162787.34
Test Set,0.820923,14.481145,10.263592,$166940.8


The finalized XGBoost model created in previous weeks performs with an $r^2$ score of 0.92. But, given that we only trained this simplified model on five features, this one works decently well. We must use a reduced model for the app because having too many input features would likely be overwhelming for the user. A model with an $r^2$ score of 0.82 is certainly worse than the finalized model, but it is perfectly serviceable for a simple app.

## Save Model and Encoder

In [38]:
# Save these variables to import into streamlit app

joblib.dump(xgboo, "model.joblib", compress=3)
joblib.dump(encoder, 'city_encoder.joblib')

['city_encoder.joblib']